In [1]:
import os
from pathlib import Path

os.getcwd()
mb_dir = Path(os.getcwd()).parent.parent.parent
os.chdir(mb_dir)
data_dir = str(mb_dir.parent / "data")
root_dir = str(mb_dir.parent)

In [2]:
import xarray as xr 
import numpy as np

from examples.paper_figures.data_utils import (
    get_model_dfs, get_plot_metrics, save_data,
    load_wyi, get_climatological_dfs, load_wyi,
    YEAR_RANGES, EXTENDED_YEARS, YEAR_RANGES_COM
)

from monsoonbench.metrics import (
    ClimatologyOnsetMetrics,
    ProbabilisticOnsetMetrics
)

from monsoonbench.visualization import create_model_comparison_table

c_metrics = ClimatologyOnsetMetrics()

# Figure 4 year ranges
YEAR_RANGES_COM = {
    "AIFS": np.arange(2004, 2022),
    "IFS": np.arange(2004, 2022),
    "FuXi": np.arange(2004, 2022),
    "Graphcast": np.arange(2004, 2022),
    "GenCast": np.arange(2019, 2022),
    "FuXi-S2S": np.arange(2004, 2022),
    "NGCM": np.arange(2004, 2022),
}

model_paths = {
    "IFS": f"{data_dir}/rainfall_4p0/IFS_S2S",
    "GenCast": f"{data_dir}/rainfall_4p0/GenCast",
    "FuXi-S2S": f"{data_dir}/rainfall_4p0/FuXi_S2S",
    "NGCM": f"{data_dir}/rainfall_4p0/NeuralGCM"
}

mp_short = {
    "GenCast": f"{data_dir}/rainfall_4p0/GenCast",
}

config = {
    "years": np.arange(2019, 2025),
    "extended_years": np.concatenate((np.arange(1965, 1979), np.arange(2019, 2025))),
    "common_years": np.arange(2004,2022),
    "imd_folder": f"{data_dir}/imd_rainfall_data/4p0",  # Ground truth rainfall data (4x4 degrees)
    "thres_file": f"{data_dir}/imd_onset_threshold/mwset4x4.nc4",  # Threshold for the onset of the monsoon (4x4 degrees)
    "shpfile_path": f"{data_dir}/ind_map_shpfile/india_shapefile.shp",  # Shapefile of India
    "output_dir": f"{root_dir}/output",  # Directory to save data files
    "file_pattern": "{}.nc"
}

MEM_NUMS = {
    "IFS": 11,
    "GenCast": 51,
    "FuXi-S2S": 51,
    "NGCM": 51,
}

def get_day_bins2(max_days):
    """Helper to generate bins based on forecast window."""
    if max_days == 15:
        return [(1, 6), (6, 11), (11, 16)]
    if max_days == 30:
        return [(1, 6), (6, 11), (11, 16), (16, 21), (21, 26), (26, 31)]
    raise ValueError(f"Unsupported max_forecast_day: {max_days}")

def get_day_bins(max_days):
    """Helper to generate bins based on forecast window."""
    if max_days == 15:
        return [(1, 5), (6, 10), (11, 15)]
    if max_days == 30:
        return [(1, 5), (6, 10), (11, 15), (16, 20), (21, 25), (26, 30)]
    raise ValueError(f"Unsupported max_forecast_day: {max_days}")

### Heatmaps

In [ ]:

def generate_heatmap_data(
    config: dict[str, str],
    model_paths: dict[str, str],
    year_ranges: dict[str, np.ndarray],
    max_forecast_day: int = 15
):
    # Initialize Metric Computation Classes
    pr = ProbabilisticOnsetMetrics()
    cl = ClimatologyOnsetMetrics()

    print("\n3. Computing climatological dataset...")
    thresh_ds = xr.open_dataset(config["thres_file"])
    thresh_slice = thresh_ds["MWmean"]
    clim_onset = cl.compute_climatological_onset_dataset(
        config["imd_folder"], thresh_slice, years=None, mok=True
    )

    # Create forecast observation dataframe
    print("\n1. Processing forecast model...")
    for model_name, model_fp in model_paths.items():
        # Model-specific parameters
        if model_name.lower() == "fuxi-s2s":
            date_filter_year=2022
        else:
            date_filter_year=2024
        if "ifs" in model_name.lower():
            mem_num=11
        else:
            mem_num=51
        
        
        #Calculate climatological skill scores
        print("\n4. Processing climatological forecasts...")
        climatology_obs_df = cl.multi_year_climatological_forecast_obs_pairs(
            clim_onset,
            target_years=year_ranges[model_name],
            model_forecast_dir=model_fp,
            mem_num=mem_num,
            max_forecast_day=max_forecast_day,
            day_bins=get_day_bins(max_forecast_day),
            date_filter_year=date_filter_year,
            file_pattern=config["file_pattern"],
            mok=True,
        )
        
        #Load and compute forcast observation pairs for each model
        forecast_obs_df = pr.multi_year_forecast_obs_pairs(
            years=year_ranges[model_name],
            imd_folder=config["imd_folder"],
            thres_file=config["thres_file"],
            model_forecast_dir=model_fp,
            mem_num=mem_num,
            max_forecast_day=max_forecast_day,
            day_bins=get_day_bins(max_forecast_day),
            date_filter_year=date_filter_year,
            file_pattern=config["file_pattern"],
            mok=True,
        )

        # Calculate brier and AUC forecast scores from observations
        print("\n2. Calculating brier and AUC forecast scores...")
        brier_forecast = pr.calculate_brier_score(forecast_obs_df)
        rps_forecast = pr.calculate_rps(forecast_obs_df)
        auc_forecast = pr.calculate_auc(forecast_obs_df)
    
        

        print("\n5. Calculating climatology scores...")
        brier_climatology = cl.calculate_brier_score_climatology(climatology_obs_df)
        rps_climatology = pr.calculate_rps(climatology_obs_df)
        auc_climatology = cl.calculate_auc_climatology(climatology_obs_df)

        # Calculate skill scores
        print("\n6. Calculating skill scores...")
        skill_results = pr.calculate_skill_scores(
            brier_forecast, rps_forecast, brier_climatology, rps_climatology
        )

        for k

        heatmap_data = {
            "skill_results": skill_results,
            "auc_forecast": auc_forecast,
            "auc_climatology": auc_climatology,
            "brier_forecast": brier_forecast,
            "brier_climatology": brier_climatology,
        }
    
    return heatmap_data

In [4]:
hm_data = generate_heatmap_data(
    config=config,
    model_paths=mp_short,
    year_ranges=YEAR_RANGES,
    max_forecast_day=30
)


3. Computing climatological dataset...
Processing 124 years: [1901, 1902, 1903, 1904, 1905, 1906, 1907, 1908, 1909, 1910, 1911, 1912, 1913, 1914, 1915, 1916, 1917, 1918, 1919, 1920, 1921, 1922, 1923, 1924, 1925, 1926, 1927, 1928, 1929, 1930, 1931, 1932, 1933, 1934, 1935, 1936, 1937, 1938, 1939, 1940, 1941, 1942, 1943, 1944, 1945, 1946, 1947, 1948, 1949, 1950, 1951, 1952, 1953, 1954, 1955, 1956, 1957, 1958, 1959, 1960, 1961, 1962, 1963, 1964, 1965, 1966, 1967, 1968, 1969, 1970, 1971, 1972, 1973, 1974, 1975, 1976, 1977, 1978, 1979, 1980, 1981, 1982, 1983, 1984, 1985, 1986, 1987, 1988, 1989, 1990, 1991, 1992, 1993, 1994, 1995, 1996, 1997, 1998, 1999, 2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]

Processing year 1901...
Loading IMD rainfall from: c:\Users\cflor\CSAssignments\Clinic3\data\imd_rainfall_data\4p0\1901.nc
Renamed dimensions: {'TIME': 'time'}
Using MOK date (June 2nd) (1901-

In [5]:
print(hm_data.keys())
hm_data["auc_forecast"]

dict_keys(['skill_results', 'auc_forecast', 'auc_climatology', 'brier_forecast', 'brier_climatology'])


{'auc': np.float64(0.8709848465648221),
 'bin_auc_scores': {'Days 1-5': np.float64(0.9373177842565598),
  'Days 6-10': np.float64(0.8558794946550049),
  'Days 11-15': np.float64(0.8430296463789962),
  'Days 16-20': np.float64(0.7907634775126584),
  'Days 21-25': np.float64(0.728440424596445),
  'Days 26-30': np.float64(0.7240201620501224),
  'After day 30': np.float64(0.9053295041553359)}}

In [14]:
print(hm_data.keys())
hm_data["auc_forecast"]

dict_keys(['skill_results', 'auc_forecast', 'auc_climatology', 'brier_forecast', 'brier_climatology'])


{'auc': np.float64(0.9412658927584301),
 'bin_auc_scores': {'Days 1-6': np.float64(0.9380443784447554),
  'Days 6-11': np.float64(0.8499652294853964),
  'Days 11-16': np.float64(0.850416282475457),
  'After day 15': np.float64(0.9362988457794466)}}

In [27]:
print(hm_data.keys())
hm_data["auc_forecast"]

bin_data = {}
full_data = {}
def get_day_tags(max_forecast_day):
    if max_forecast_day == 15:
        return [f"d{low}_{high}" for low, high in get_day_bins(15)]
    elif max_forecast_day == 30:
        return [f"d{low}_{high}" for low, high in get_day_bins(30)]
    else:
        raise ValueError(f"Unsupported max_forecast_day: {max_forecast_day}")
day_tags_30 = get_day_tags(30)
day_tags_15 = get_day_tags(15)
max_forecast_day = 30

def get_col_tag(text):
    if "skill_results" in text:
        return "fair_brier_skill"
    elif "auc" in text:
        return "auc"
    elif "brier" in text:
        return "fair_brier"

def get_model_tag(model_name):
    if "ifs" in model_name.lower():
        return "ifss2s"
    if "fuxi" in model_name.lower():
        return "fuxis2s"
    if "gen" in model_name.lower():
        return "gencast"
    if "ngcm" in model_name.lower():
        return "ngcm"

#climatology 
if "model_label" not in bin_data:
    bin_data["model_label"] = []
if "horizon" not in bin_data:
    bin_data["horizon"] = []
bin_data["model_label"].append("clim")
bin_data["horizon"].append(str(max_forecast_day))

for key, value in hm_data.items():
    if "clim" not in key:
        continue
    base_tag = get_col_tag(key)
    print(base_tag)
    bin_key = [x for x in value.keys() if "bin" in x][0]
    for bin_tag, bin_score in zip(get_day_tags(max_forecast_day), value[bin_key].values()):
        tag = f"{base_tag}_{bin_tag}"
        if tag not in bin_data:
            bin_data[tag] = []
        bin_data[tag].append(bin_score)
        print(tag, bin_score)

model_name="IFS"
model_tag = get_model_tag(model_name)
bin_data["model_label"].append(model_tag)
bin_data["horizon"].append(str(max_forecast_day))

for key, value in hm_data.items():
    if "clim" in key:
        continue
    base_tag = get_col_tag(key)
    print(base_tag)
    bin_key = [x for x in value.keys() if "bin" in x][0]
    for bin_tag, bin_score in zip(get_day_tags(max_forecast_day), value[bin_key].values()):
        tag = f"{base_tag}_{bin_tag}"
        if tag not in bin_data:
            bin_data[tag] = []
        bin_data[tag].append(bin_score)
        print(tag, bin_score)
    

# for key, value in hm_data.items():
#     base_tag = get_col_tag(key)
#     if "auc" in key:
#         cat_keys = list(hm_data.keys())
#         print(cat_keys)
#         bins = value

#hm_data

dict_keys(['skill_results', 'auc_forecast', 'auc_climatology', 'brier_forecast', 'brier_climatology'])
auc
auc_d1_5 0.8801749271137026
auc_d6_10 0.8383219954648526
auc_d11_15 0.7884712015576911
auc_d16_20 0.7962653473210445
auc_d21_25 0.7684661767476917
auc_d26_30 0.7757835563092771
fair_brier
fair_brier_d1_5 0.10048080775035778
fair_brier_d6_10 0.07918233733934628
fair_brier_d11_15 0.08616877162577943
fair_brier_d16_20 0.08132077920043972
fair_brier_d21_25 0.08735686674017445
fair_brier_d26_30 0.08683874013096292
fair_brier_skill
fair_brier_skill_d1_5 0.3709406658860609
fair_brier_skill_d6_10 0.10541922520151936
fair_brier_skill_d11_15 0.09098309010452743
fair_brier_skill_d16_20 -0.009297756131396273
fair_brier_skill_d21_25 -0.04374882508340572
fair_brier_skill_d26_30 -0.019993499228659672
auc
auc_d1_5 0.9373177842565598
auc_d6_10 0.8558794946550049
auc_d11_15 0.8430296463789962
auc_d16_20 0.7907634775126584
auc_d21_25 0.728440424596445
auc_d26_30 0.7240201620501224
fair_brier
fair_br

In [28]:
bin_data

{'model_label': ['clim', 'ifss2s'],
 'horizon': ['30', '30'],
 'auc_d1_5': [np.float64(0.8801749271137026), np.float64(0.9373177842565598)],
 'auc_d6_10': [np.float64(0.8383219954648526), np.float64(0.8558794946550049)],
 'auc_d11_15': [np.float64(0.7884712015576911),
  np.float64(0.8430296463789962)],
 'auc_d16_20': [np.float64(0.7962653473210445),
  np.float64(0.7907634775126584)],
 'auc_d21_25': [np.float64(0.7684661767476917), np.float64(0.728440424596445)],
 'auc_d26_30': [np.float64(0.7757835563092771),
  np.float64(0.7240201620501224)],
 'fair_brier_d1_5': [0.10048080775035778, 0.13015328470922474],
 'fair_brier_d6_10': [0.07918233733934628, 0.05010658528536268],
 'fair_brier_d11_15': [0.08616877162577943, 0.07884541448555289],
 'fair_brier_d16_20': [0.08132077920043972, 0.08271684742272978],
 'fair_brier_d21_25': [0.08735686674017445, 0.09198631689981171],
 'fair_brier_d26_30': [0.08683874013096292, 0.0895076985042383],
 'fair_brier_skill_d1_5': [0.3709406658860609],
 'fair_bri

In [29]:
hm_data

{'skill_results': {'fair_brier_skill_score': np.float64(0.01793890485742522),
  'fair_rps_skill_score': np.float64(-0.043066402225265143),
  'bin_fair_brier_skill_scores': {'Days 1-5': 0.3709406658860609,
   'Days 6-10': 0.10541922520151936,
   'Days 11-15': 0.09098309010452743,
   'Days 16-20': -0.009297756131396273,
   'Days 21-25': -0.04374882508340572,
   'Days 26-30': -0.019993499228659672}},
 'auc_forecast': {'auc': np.float64(0.8709848465648221),
  'bin_auc_scores': {'Days 1-5': np.float64(0.9373177842565598),
   'Days 6-10': np.float64(0.8558794946550049),
   'Days 11-15': np.float64(0.8430296463789962),
   'Days 16-20': np.float64(0.7907634775126584),
   'Days 21-25': np.float64(0.728440424596445),
   'Days 26-30': np.float64(0.7240201620501224),
   'After day 30': np.float64(0.9053295041553359)}},
 'auc_climatology': {'auc': np.float64(0.8617383837164057),
  'bin_auc_scores': {'Days 1-5': np.float64(0.8801749271137026),
   'Days 6-10': np.float64(0.8383219954648526),
   'Days